In [35]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [36]:
d_model=512
num_heads=8
num_layers=6
seq_len=10
vocab_size=5000
batch_size=2

In [37]:
input_ids=torch.randint(
    0,
    vocab_size,
    (batch_size,seq_len)
)
print(input_ids)

tensor([[3034, 4944, 3332,  521, 1169, 2625, 4935,  575, 3076, 1088],
        [ 336, 1518, 3670, 3874, 2703,  647, 3085, 4920, 1671,  892]])


In [38]:
token_embedding=nn.Embedding(
    vocab_size,
    d_model
)
x=token_embedding(input_ids)

In [39]:
position_embedding=nn.Embedding(
    seq_len,
    d_model
)
positions=torch.arange(seq_len)
positions=positions.unsqueeze(0)
position_vectors=position_embedding(positions)

In [40]:
x=x+position_vectors

In [41]:
class GPTBlock(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):   

        attention_output, _ = self.attention(
            query=x,
            key=x,
            value=x,
            attn_mask=mask
        )

        x = self.norm1(x + attention_output)

        ffn_output = self.ffn(x)

        x = self.norm2(x + ffn_output)

        return x

In [42]:
blocks = nn.ModuleList(
    [
        GPTBlock(d_model, num_heads)
        for _ in range(num_layers)
    ]
)

In [43]:
mask = torch.triu(
    torch.ones(seq_len, seq_len),
    diagonal=1
)

mask = mask.masked_fill(mask == 1, float("-inf"))
print(mask)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


In [44]:
for block in blocks:
    x = block(x, mask)

In [45]:
final_norm = nn.LayerNorm(d_model)

x = final_norm(x)

In [46]:
lm_head = nn.Linear(
    d_model,
    vocab_size
)
logits = lm_head(x)

In [47]:
probabilities = F.softmax(
    logits,
    dim=-1
)

In [48]:
next_token = torch.argmax(
    probabilities,
    dim=-1
)

print(next_token)

tensor([[2069, 3740, 4411, 1492, 3856,  407, 4719,  253, 3117,  486],
        [ 952,  952, 1330, 3129, 4034, 3129, 2173, 2596, 4078, 2778]])


In [49]:
targets = torch.randint(
    0,
    vocab_size,
    (batch_size, seq_len)
)

loss = F.cross_entropy(
    logits.view(-1, vocab_size),
    targets.view(-1)
)

print("Loss:", loss.item())

Loss: 8.592397689819336
